# Homework 3 — Machine Learning Zoomcamp 2026

Решение всех шести вопросов на официальном датасете.

[Условия](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/03-classification/homework.md) · [Проверочный код курса](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/data/validate_homework.py)

Запускайте ячейки сверху вниз. Использованы NumPy 2.3.3, Pandas 2.3.2 и Scikit-Learn 1.7.2. Данные загружаются автоматически, контрольная сумма проверяется.

In [1]:
# При необходимости установите зависимости и перезапустите kernel:
# %pip install numpy==2.3.3 pandas==2.3.2 scikit-learn==1.7.2

import hashlib
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

print('NumPy:', np.__version__, 'Pandas:', pd.__version__, 'Scikit-Learn:', sklearn.__version__)

NumPy: 2.3.3 Pandas: 2.3.2 Scikit-Learn: 1.7.2


In [2]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
path = Path('course_lead_scoring_2026.csv')
if not path.exists():
    with urlopen(url) as response:
        path.write_bytes(response.read())
raw = path.read_bytes()
expected_sha256 = '2d9da196bdefd2a45aa0a17ca21c19f06a5d9d56af628be1f04bbb3531cc12e9'
assert hashlib.sha256(raw).hexdigest() == expected_sha256, 'Датасет не соответствует официальному релизу'
df = pd.read_csv(BytesIO(raw))
print('Размер:', df.shape)
print('Пропуски до заполнения:')
print(df.isna().sum().to_string())
categorical = df.drop(columns='converted').select_dtypes(include='object').columns.tolist()
numerical = [c for c in df.columns if c not in categorical and c != 'converted']
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

Размер: (5000, 9)
Пропуски до заполнения:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0


## Вопрос 1 — мода industry

In [3]:
counts = df.industry.value_counts()
q1 = counts.index[0]
print(counts.to_string())
print('Ответ:', q1, '— вариант 2')

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Ответ: technology — вариант 2


## Вопрос 2 — корреляция числовых признаков

Сравниваем только четыре пары из условий.

In [4]:
corr = df[numerical].corr()
pairs = [('interaction_count', 'lead_score'), ('number_of_courses_viewed', 'lead_score'), ('number_of_courses_viewed', 'interaction_count'), ('annual_income', 'interaction_count')]
pair_scores = {f'{a} + {b}': corr.loc[a, b] for a, b in pairs}
print(pd.Series(pair_scores).to_string(float_format=lambda x: f'{x:.15f}'))
q2 = max(pair_scores, key=lambda name: abs(pair_scores[name]))
print('Ответ:', q2, '— вариант 1')

interaction_count + lead_score                 0.915745798041870
number_of_courses_viewed + lead_score          0.757204239210489
number_of_courses_viewed + interaction_count   0.721609003893704
annual_income + interaction_count              0.122842351359560
Ответ: interaction_count + lead_score — вариант 1


## Разбиение 60% / 20% / 20%

Оба вызова используют random_state=42. Target исключается из признаков.

In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
y_train = df_train.converted.to_numpy()
y_val = df_val.converted.to_numpy()
y_test = df_test.converted.to_numpy()
df_train = df_train.drop(columns='converted')
df_val = df_val.drop(columns='converted')
df_test = df_test.drop(columns='converted')
print('Train / validation / test:', len(df_train), len(df_val), len(df_test))

Train / validation / test: 3000 1000 1000


## Вопрос 3 — mutual information

Используем только обучающую выборку и mutual_info_score.

In [6]:
mi = {c: mutual_info_score(y_train, df_train[c]) for c in categorical}
for feature, value in mi.items():
    print(feature, repr(value), 'округление:', round(value, 2))
q3 = max(mi, key=mi.get)
print('Ответ:', q3, '— вариант 3')

lead_source 0.03434605791617973 округление: 0.03
industry 0.0019945636715935217 округление: 0.0
employment_status 0.018965604906982875 округление: 0.02
location 0.001042763579701525 округление: 0.0
Ответ: lead_source — вариант 3


## Вопрос 4 — логистическая регрессия

DictVectorizer обучается только на train. Все признаки используются без масштабирования. Порог классификации — 0.5.

In [7]:
def evaluate(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

features = df_train.columns.tolist()
baseline = evaluate(features)
print('Accuracy:', baseline)
print('Ответ:', round(baseline, 2), '— вариант 2')

Accuracy: 0.645
Ответ: 0.65 — вариант 2


## Вопрос 5 — исключение признаков

Для каждой модели заново обучаем DictVectorizer и LogisticRegression. Показываем разницу со знаком и модуль. Официальный validate_homework.py выбирает наименьший модуль среди предложенных вариантов.

In [8]:
rows = []
for feature in features:
    accuracy = evaluate([c for c in features if c != feature])
    difference = baseline - accuracy
    rows.append({'excluded': feature, 'accuracy': accuracy, 'difference': difference, 'absolute_difference': abs(difference)})
elimination = pd.DataFrame(rows).set_index('excluded')
print(elimination.to_string(float_format=lambda x: f'{x:.3f}'))
candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
q5 = elimination.loc[candidates, 'absolute_difference'].idxmin()
print('Ответ:', q5, '— вариант 2')

                          accuracy  difference  absolute_difference
excluded                                                           
lead_source                  0.642       0.003                0.003
industry                     0.645       0.000                0.000
employment_status            0.645       0.000                0.000
location                     0.645       0.000                0.000
annual_income                0.724      -0.079                0.079
number_of_courses_viewed     0.643       0.002                0.002
interaction_count            0.601       0.044                0.044
lead_score                   0.644       0.001                0.001
Ответ: number_of_courses_viewed — вариант 2


## Вопрос 6 — регуляризация

При одинаковой accuracy выбираем наименьший C.

In [9]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]
c_scores = {C: evaluate(features, C=C) for C in c_values}
for C, accuracy in c_scores.items():
    print('C =', C, 'accuracy =', accuracy, 'round(accuracy, 3) =', round(accuracy, 3))
q6 = max(c_values, key=lambda C: (round(c_scores[C], 3), -C))
print('Ответ:', q6, '— вариант 4')

C = 1e-06 accuracy = 0.598 round(accuracy, 3) = 0.598
C = 1e-05 accuracy = 0.598 round(accuracy, 3) = 0.598
C = 0.0001 accuracy = 0.613 round(accuracy, 3) = 0.613
C = 0.001 accuracy = 0.645 round(accuracy, 3) = 0.645
Ответ: 0.001 — вариант 4


## Итоговые ответы

In [10]:
answers = [q1, q2, q3, round(baseline, 2), q5, q6]
options = [2, 1, 3, 2, 2, 4]
for question, (answer, option) in enumerate(zip(answers, options), 1):
    print(f'{question}. {answer} — вариант {option}')

1. technology — вариант 2
2. interaction_count + lead_score — вариант 1
3. lead_source — вариант 3
4. 0.65 — вариант 2
5. number_of_courses_viewed — вариант 2
6. 0.001 — вариант 4
